<a href="https://colab.research.google.com/github/sriharikrishna/JDEV-Tutorial/blob/python/05Reverse/05Reverse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Converting C code to JAX

This section re-implements the provided C code in JAX, a Python library for high-performance numerical computing and automatic differentiation. The goal is to replicate the C logic for calculating 'total affinity' and then leverage JAX's capabilities to compute gradients of this affinity with respect to the input parameters.

Key aspects of the conversion:

1.  **Functional Programming**: JAX encourages a functional style. Instead of modifying data structures in-place (like `pts[i].aff *= ...` in C), we use JAX's immutable arrays and functional updates (e.g., `affs.at[idx].multiply(sim)`).
2.  **Vectorization (`jax.vmap`)**: Pairwise distance and similarity calculations are efficiently vectorized using `jax.vmap` to avoid explicit Python loops over large arrays, which would be slow and non-differentiable in JAX.
3.  **Looping (`jax.lax.scan`)**: The C code's iteration over `connections` to update point affinities is translated into `jax.lax.scan`. This is JAX's way to perform functional, differentiable loops.
4.  **Automatic Differentiation (`jax.grad`)**: The primary benefit of JAX is demonstrated by using `jax.grad` to automatically compute the gradient of the final `total_affinity` with respect to the input point coordinates, `sigma`, and `max_val`.

In [ ]:
import jax
import jax.numpy as jnp

# Helper function to calculate squared Euclidean distance between two 3D points
def norm_pts_jax(pt1_coords, pt2_coords):
    """Calculates squared Euclidean distance between two points."""
    diff = pt1_coords - pt2_coords
    return jnp.sum(diff * diff)

# Helper function to compute similarity based on distance and sigma
def compute_similarity_jax(d, sigma):
    """Computes similarity based on distance and sigma."""
    return jnp.exp(-d / sigma)


In [ ]:
def compute_total_affinity(pts_coords, sigma, max_val, connections_indices):
    """
    Reimplementation of the C compute_matrices logic to return total squared affinity.

    Args:
        pts_coords (jnp.ndarray): (nb_pts, 3) array of point coordinates.
        sigma (float): Sigma parameter for similarity calculation.
        max_val (float): Max value for distance normalization.
        connections_indices (jnp.ndarray): (num_connections, 2) array of connection indices.

    Returns:
        float: The sum of squared affinities of all points.
    """
    nb_pts = pts_coords.shape[0]

    # Compute all pairwise normalized squared distances (A matrix)
    # Using vmap to vectorize norm_pts_jax for all pairs
    pairwise_norm_pts_fn = jax.vmap(jax.vmap(norm_pts_jax, in_axes=(None, 0)), in_axes=(0, None))
    A_raw = pairwise_norm_pts_fn(pts_coords, pts_coords)
    A = A_raw / (max_val * max_val)

    # Compute similarity matrix (S matrix)
    S = compute_similarity_jax(A, sigma)

    # Initialize affinities (equivalent to pts[i].aff = 1 in C)
    # Using float32 for compatibility with C's `float` type if needed for direct comparison.
    current_affinities = jnp.ones(nb_pts, dtype=jnp.float32)

    # Define a scan function to update affinities for a single connection
    def update_affinity_for_connection(affs, conn_pair):
        idx1, idx2 = conn_pair
        sim = S[idx1, idx2]
        # Multiply affinities of connected points by their similarity
        affs = affs.at[idx1].multiply(sim)
        affs = affs.at[idx2].multiply(sim)
        return affs, None # Changed to return a pair (carry, y)

    # Use jax.lax.scan to iterate over connections and update affinities functionally
    # `scan` returns (final_carry, final_stacked_outputs). We only need the final_carry (affinities).
    final_affinities = jax.lax.scan(update_affinity_for_connection, current_affinities, connections_indices)[0]

    # Calculate total affinity (sum of squared final affinities)
    total_aff = jnp.sum(final_affinities * final_affinities)
    return total_aff


### Data Setup and Execution (Mirroring C's `main` function)

Here, we set up the initial point coordinates and connections as defined in the C code's `main` function. We then calculate the total affinity using our JAX function and compute its gradients.

In [ ]:
# C main function equivalent setup
nb_pts = 5
sigma_val = 20.0
max_val = 10.0

# Initial point coordinates (from C code)
initial_pts_coords = jnp.array([
    [1.0, 1.0, 1.0],  # pt[0]
    [0.0, 1.0, 1.0],  # pt[1]
    [0.0, 0.0, 1.0],  # pt[2]
    [-1.0, -1.0, 1.0], # pt[3]
    [-1.0, -1.0, -1.0] # pt[4]
], dtype=jnp.float32) # Ensure float32 for consistency with C's `float`

# Generate connections_indices matching the C code's processing order.
# The C code builds a linked list by prepending new connections, so the processing
# order is the reverse of the creation order.
connections_creation_order = []
for i in range(nb_pts):
    current_j = i + 1
    while current_j < nb_pts:
        connections_creation_order.append((i, current_j))
        current_j += (i + 1)

# Reverse the creation order to get the actual order in which the C code processes connections
connections_indices_jax = jnp.array(list(reversed(connections_creation_order)), dtype=jnp.int32)

# --- Execution and Gradient Computation ---

# Calculate the total affinity using the JAX function
total_aff_jax = compute_total_affinity(initial_pts_coords, sigma_val, max_val, connections_indices_jax)
print(f"JAX Total affinity = {total_aff_jax:.6f}")

# Compute gradients of the total affinity with respect to point coordinates, sigma, and max_val
grad_fn = jax.grad(compute_total_affinity, argnums=(0, 1, 2)) # 0: pts_coords, 1: sigma, 2: max_val
grads_pts, grads_sigma, grads_max_val = grad_fn(initial_pts_coords, sigma_val, max_val, connections_indices_jax)

print("\n--- Gradients of Total Affinity ---")
print("Gradient with respect to pts_coords (∂aff/∂x, ∂aff/∂y, ∂aff/∂z for each point):")
print(grads_pts)
print("\nGradient with respect to sigma (∂aff/∂sigma):")
print(grads_sigma)
print("\nGradient with respect to max_val (∂aff/∂max_val):")
print(grads_max_val)


JAX Total affinity = 4.922742

--- Gradients of Total Affinity ---
Gradient with respect to pts_coords (∂aff/∂x, ∂aff/∂y, ∂aff/∂z for each point):
[[-2.3540923e-02 -1.9608278e-02 -7.8102890e-03]
 [ 3.9055012e-06 -1.1825625e-02 -7.8574801e-03]
 [-1.7670449e-05  3.9504752e-03  0.0000000e+00]
 [ 1.1815660e-02  1.1815660e-02 -7.8456290e-03]
 [ 1.1739029e-02  1.5667770e-02  2.3513399e-02]]

Gradient with respect to sigma (∂aff/∂sigma):
0.0038259933

Gradient with respect to max_val (∂aff/∂max_val):
0.015303975
